In [6]:
from needle_train.augmentor_pipeline import *
from utils import *
from config import *
from preprocessing import *
from training import *
from needle_train.build_data import *

In [2]:
# some data storage 
for t in ['hosted_set', 'hostless_set']:
    '''
    combine the data from two paths from .npy files. Two files are two sets of cross-matched data. 
    datapath1 is the data with min_detection_1, datapath2 is the data with min_detection_2.
    save_path is the combined data.
    '''
    for label in ['SLSN-I', 'TDE']:
        data_path1 = '/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_inputs/' + t + '/' + label + '/data_dict_min_detection_1.npy'
        data_path2 = '/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_inputs/' + t + '/' + label + '/data_dict.npy'
        save_path = '/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_inputs/' + t + '/' + label + '/data_dict_crossmatched_mask.npy'
        combine_np_data(data_path1, data_path2, save = True, save_path = save_path)

data1:  (2841, 2, 60, 60) (2841, 26) (2841,) (2841,) (2841, 2)
data2:  (3276, 2, 60, 60) (3276, 26) (3276,) (3276,) (3276, 2)
combined:  (6117, 2, 60, 60) (6117, 26) (6117,) (6117,) (6117, 2)
data1:  (4142, 2, 60, 60) (4142, 26) (4142,) (4142,) (4142, 2)
data2:  (5057, 2, 60, 60) (5057, 26) (5057,) (5057,) (5057, 2)
combined:  (9199, 2, 60, 60) (9199, 26) (9199,) (9199,) (9199, 2)
data1:  (3006, 2, 60, 60) (3006, 16) (3006,) (3006,) (3006, 2)
data2:  (3526, 2, 60, 60) (3526, 16) (3526,) (3526,) (3526, 2)
combined:  (6532, 2, 60, 60) (6532, 16) (6532,) (6532,) (6532, 2)
data1:  (0,) (0,) (0,) (0,) (0,)
data2:  (0,) (0,) (0,) (0,) (0,)
combined:  (0,) (0,) (0,) (0,) (0,)


In [2]:
for t in ['hosted_set', 'hostless_set']:
    '''
    combine the data from two paths from .npy files.  
    datapath1 is the data with crossmatched and masked, datapath2 is the original data with masked.
    save_path is the combined data. Add them together for the full training set. 
    '''
    for label in ['SLSN-I', 'TDE']:
        data_path1 = '/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_inputs/' + t + '/' + label + '/data_dict_original_mask.npy'
        data_path2 = '/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_inputs/' + t + '/' + label + '/data_dict_crossmatched_mask.npy'
        save_path = '/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_inputs/' + t + '/' + label + '/data_dict_full_mask.npy'
        combine_np_data(data_path1, data_path2, save = True, save_path = save_path)

data1:  (37, 2, 60, 60) (37, 26) (37,) (37,) (37, 2)
data2:  (6117, 2, 60, 60) (6117, 26) (6117,) (6117,) (6117, 2)
combined:  (6154, 2, 60, 60) (6154, 26) (6154,) (6154,) (6154, 2)
data1:  (31, 2, 60, 60) (31, 26) (31,) (31,) (31, 2)
data2:  (9199, 2, 60, 60) (9199, 26) (9199,) (9199,) (9199, 2)
combined:  (9230, 2, 60, 60) (9230, 26) (9230,) (9230,) (9230, 2)
data1:  (41, 2, 60, 60) (41, 16) (41,) (41,) (41, 2)
data2:  (6532, 2, 60, 60) (6532, 16) (6532,) (6532,) (6532, 2)
combined:  (6573, 2, 60, 60) (6573, 16) (6573,) (6573,) (6573, 2)
No data to combine, save empty file


In [ ]:
for t in ['hosted_set', 'hostless_set']:
    image_full = []
    metaset_full = []
    labels_full = []
    idx_set_full = []

    idx = 0
    for label in ['SLSN-I', 'TDE']:
        data_path = '/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_inputs/' + t + '/' + label + '/data_dict_full_mask.npy'
        if os.path.exists(data_path) is False:
            continue
        imageset, metaset, labels, z_set, obj_match_set = load_np_data(data_path)
        if imageset.shape[0] == 0:
            continue
        image_full.append(imageset)
        metaset_full.append(metaset)
        labels_full.append(labels)
        idx_set_full.append(np.arange(idx, idx + imageset.shape[0]))
        idx += imageset.shape[0]

     
        
    SN_data_path = '/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_inputs/' + t + '/SN/data_dict_original_mask.npy'
    imageset, metaset, labels, z_set, obj_match_set = load_np_data(SN_data_path)
    image_full.append(imageset)
    metaset_full.append(metaset)
    labels_full.append(labels)
    idx_set_full.append(np.arange(idx, idx + imageset.shape[0]))
    idx += imageset.shape[0]

    image_full = np.concatenate(image_full, axis=0).astype(np.float32)
    metaset_full = np.concatenate(metaset_full, axis=0).astype(np.float32)
    labels_full = np.concatenate(labels_full, axis=0)
    idx_set_full = np.concatenate(idx_set_full, axis=0)
    
    metaset_full[np.isnan(metaset_full)] = 0

    sci_images = image_full[:,0,:,:]  # Shape (4363, 60, 60)
    sci_images = sci_images[..., np.newaxis]  # Add channel dimension to make shape (N, 60, 60, 1)
    ref_images = image_full[:,1,:,:]  # Shape (4363, 60, 60)
    ref_images = ref_images[..., np.newaxis]
    # Concatenate along a new axis to get shape (4363, 60, 120)
    image_full = np.concatenate([sci_images, ref_images], axis=-1)

    print(image_full.shape, metaset_full.shape, labels_full.shape, idx_set_full.shape)
    save_to_h5py(image_full, metaset_full, labels_full, idx_set_full, '/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_inputs/' + t + '/training_set_full_mask.hdf5')







    

(19837, 60, 60, 2) (19837, 26) (19837,) (19837,)
Image shape:  (19837, 60, 60, 2)
Meta shape:  (19837, 26)
Label shape:  (19837,)
Saved to /Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_inputs/hosted_set/training_set_full_mask.hdf5
(6767, 60, 60, 2) (6767, 16) (6767,) (6767,)
Image shape:  (6767, 60, 60, 2)
Meta shape:  (6767, 16)
Label shape:  (6767,)
Saved to /Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_inputs/hostless_set/training_set_full_mask.hdf5


In [ ]:
for t in ['hosted_set', 'hostless_set']:
    image_full = []
    metaset_full = []
    labels_full = []
    idx_set_full = []

    idx = 0
    for label in ['SLSN-I', 'TDE', 'SN']:
        data_path = '/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_inputs/' + t + '/' + label + '/data_dict_original_mask.npy'
        if os.path.exists(data_path) is False:
            continue
        imageset, metaset, labels, z_set, obj_match_set = load_np_data(data_path)
        if imageset.shape[0] == 0:
            continue
        image_full.append(imageset)
        metaset_full.append(metaset)
        labels_full.append(labels)
        idx_set_full.append(np.arange(idx, idx + imageset.shape[0]))
        idx += imageset.shape[0]

    

    image_full = np.concatenate(image_full, axis=0).astype(np.float32)
    metaset_full = np.concatenate(metaset_full, axis=0).astype(np.float32)
    labels_full = np.concatenate(labels_full, axis=0)
    idx_set_full = np.concatenate(idx_set_full, axis=0)
    
    metaset_full[np.isnan(metaset_full)] = 0

    sci_images = image_full[:,0,:,:]  # Shape (4363, 60, 60)
    sci_images = sci_images[..., np.newaxis]  # Add channel dimension to make shape (N, 60, 60, 1)
    ref_images = image_full[:,1,:,:]  # Shape (4363, 60, 60)
    ref_images = ref_images[..., np.newaxis]
    # Concatenate along a new axis to get shape (4363, 60, 120)
    image_full = np.concatenate([sci_images, ref_images], axis=-1)

    print(image_full.shape, metaset_full.shape, labels_full.shape, idx_set_full.shape)
    save_to_h5py(image_full, metaset_full, labels_full, idx_set_full, '/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_inputs/' + t + '/training_set_original_mask.hdf5')

(4521, 60, 60, 2) (4521, 26) (4521,) (4521,)
Image shape:  (4521, 60, 60, 2)
Meta shape:  (4521, 26)
Label shape:  (4521,)
Saved to /Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_inputs/hosted_set/training_set_original_mask.hdf5
(236, 60, 60, 2) (236, 16) (236,) (236,)
Image shape:  (236, 60, 60, 2)
Meta shape:  (236, 16)
Label shape:  (236,)
Saved to /Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_inputs/hostless_set/training_set_original_mask.hdf5


In [39]:
# for t in ['hosted_set', 'hostless_set']:
#     host_path = os.path.join(data_path, t)
#     dataset, metaset, labels, idx_set = convert_untouched_data(host_path)
#     save_to_h5py(dataset, metaset, labels, idx_set, host_path + '/untouched_set.hdf5')
 

/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_input_v2/hosted_set/untouched/data_dict_full.npy
(15, 60, 60, 2)
(15,) (15, 26) (15,)
Image shape:  (15, 60, 60, 2)
Meta shape:  (15, 26)
Label shape:  (15,)
/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_input_v2/hostless_set/untouched/data_dict_full.npy
(17, 60, 60, 2)
(17,) (17, 16) (17,)
Image shape:  (17, 60, 60, 2)
Meta shape:  (17, 16)
Label shape:  (17,)


In [14]:
untouched_path = '/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/untouched_2025/20240225_20250603.csv'
data = pd.read_csv(untouched_path)
types = list(set(data['type']))

raw_label_dict = {}
for label in types:
    if label == 'TDE' or label == 'TDE-He':
        raw_label_dict[label] = 2
    elif label == 'SLSN-I':
        raw_label_dict[label] = 1
    elif label == 'nova' or label == 'LBV' or label == 'LRN' or label == 'SN Ca-rich-Ca' or label == 'Gap':
        raw_label_dict[label] = 3
    else:
        raw_label_dict[label] = 0


print(raw_label_dict)



{'SLSN-II': 0, 'SN Ib/c': 0, 'SN Ib-pec': 0, 'SN Ca-rich-Ca': 3, 'SN IIP': 0, 'SN Ia': 0, 'SN Ic-pec': 0, 'SN II': 0, 'SN II-pec': 0, 'SN Ia-CSM': 0, 'SLSN-I': 1, 'SN Ibn': 0, 'LBV': 3, 'SN Ic-BL': 0, 'SN IIb': 0, 'SN Iax': 0, 'SN Ic': 0, 'nova': 3, 'SN Ia-91T': 0, 'SN Ia-91bg': 0, 'SN Ib': 0, 'SN IIn': 0, 'LRN': 3, 'TDE-He': 2, 'SN Ia-SC': 0, 'SN Ia-pec': 0, 'SN Icn': 0, 'TDE': 2}


In [3]:
from build_data import *
for t in ['hosted_set', 'hostless_set']:
    data_path = '/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_inputs/' + t + '/'
    imageset, metaset, labels, idx_set = convert_untouched_data(data_path, data_type = 'original_mask_2025', save = True)

/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_inputs/hosted_set/untouched/data_dict_original_mask_2025.npy
(2421, 60, 60, 2) (2421, 26) (2421,) (2421,)
Image shape:  (2421, 60, 60, 2)
Meta shape:  (2421, 26)
Label shape:  (2421,)
Saved to /Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_inputs/hosted_set//untouched_set_original_mask_2025.hdf5
/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_inputs/hostless_set/untouched/data_dict_original_mask_2025.npy
(172, 60, 60, 2) (172, 16) (172,) (172,)
Image shape:  (172, 60, 60, 2)
Meta shape:  (172, 16)
Label shape:  (172,)
Saved to /Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_inputs/hostless_set//untouched_set_original_mask_2025.hdf5


In [7]:
from build_data import *
for t in ['hosted_set', 'hostless_set']:
    data_path = '/Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_inputs/' + t + '/'
    # imageset, metaset, labels, idx_set = convert_untouched_data(data_path, data_type = 'original_mask_2025', save = True)
    imageset1, metaset1, labels1, idx_set1 = open_with_h5py(data_path + 'untouched_set_original_mask_2025.hdf5')
    imageset2, metaset2, labels2, idx_set2 = open_with_h5py(data_path + 'untouched_set_original_mask_2024.hdf5')

    imageset = np.concatenate([imageset1, imageset2], axis=0)
    labels = np.concatenate([labels1, labels2], axis=0)
    metaset = np.concatenate([metaset1, metaset2], axis=0)
    idx_set = np.concatenate([idx_set1, idx_set2], axis=0)
    print(imageset.shape, labels.shape, metaset.shape, idx_set.shape)
    save_to_h5py(imageset, metaset, labels, idx_set, data_path + 'untouched_set_original_mask_full.hdf5')
    

(2436, 60, 60, 2) (2436,) (2436, 26) (2436,)
Image shape:  (2436, 60, 60, 2)
Meta shape:  (2436, 26)
Label shape:  (2436,)
Saved to /Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_inputs/hosted_set/untouched_set_original_mask_full.hdf5
(189, 60, 60, 2) (189,) (189, 16) (189,)
Image shape:  (189, 60, 60, 2)
Meta shape:  (189, 16)
Label shape:  (189,)
Saved to /Users/xinyuesheng/Documents/astro_projects/scripts/NEEDLE2.0/needle_inputs/hostless_set/untouched_set_original_mask_full.hdf5


In [4]:
imageset.min(), imageset.max(), imageset.mean(), imageset.std()



(0.0, 1.0, 0.06262559, 0.065289006)